# Task Decomposition: CallMeMaybe Operator Performance

This notebook contains the step-by-step guide that will be used to achieve the project's main objective: identify underperforming agents and test statistical hypotheses about their performance.

## Table of contents

1. Business problem
2. Initial data assessment
3. Hypotheses
4. Key performance indicators (KPIs)
5. Action plan
6. Stakeholders
7. Summary

## 1. Business problem

CallMeMaybe provides a cloud telephony service to companies. This kind of company usually provides its clients with features such as telephone numbers, call forwarding and an app where agents can manage their calls. Inherent to these services, it collects relevant daily call statistics that, when properly processed and analyzed, can generate meaningful information.

To use that information, CallMeMaybe is developing a new feature that will allow its clients' supervisors to identify underperforming agents based on unanswered calls, waiting time on inbound calls and the number of calls made by outbound agents.

Underperforming agents affect both the people who call and the client companies. Long waiting times and missed calls are among the factors that most affect customer retention: a caller who is not answered quickly may choose a competitor instead of the company they first tried to contact. In addition, each missed call may represent a lost sale, booking or request, directly reducing the client company's revenue.

Agents with low productivity also increase costs for the client company. Outbound agents who make few calls increase the cost per contact, and agents with long waiting times or many missed calls deliver less for the same cost. For example, a company pays the same salary to an agent who handles 20 calls as to one who handles 5 (illustrative figures).

With the development of this new feature, CallMeMaybe may move closer to providing a Contact Center as a Service (CCaaS) solution. The information provided could give it an advantage over its competitors, as its clients will be able to track agent performance and define action plans to improve their KPIs. As a result, CallMeMaybe may increase client retention and grow its revenue by attracting more clients, as well as clients on higher-value plans.

To classify agents as underperforming, some aspects of the definition must first be clarified:

- **Comparison group:** data should be grouped by client company, so that agents are only compared with agents working in similar conditions.
- **Missed calls:** the share of missed calls over total inbound calls is more informative than the absolute number, since agents handle different call volumes.
- **Waiting time:** it must be measured per call, and it remains to be defined whether it should be compared within each company or against an industry benchmark.
- **Outbound agents:** since the data does not identify agent roles, this must be defined based on the data. Possible criteria are agents who only make outbound calls, or agents with a high volume of outbound calls when compared to the average of their company.

This project will deliver the metrics and criteria needed to identify underperforming agents, supported by statistical hypothesis testing and presented in a dashboard and a final report, so that supervisors can intervene and improve agent performance.

## 2. Initial data assessment

In [1]:
# Import necessary libraries
import pandas as pd

In [2]:
# Read the CSV files into DataFrames
df_calls = pd.read_csv('../data/telecom_dataset_new.csv')
df_clients = pd.read_csv('../data/telecom_clients.csv')

In [3]:
def data_overview(df):
    """
    Display an overview of a DataFrame.

    Shows the first rows, shape, data types, descriptive statistics,
    duplicate rows and missing values.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame to inspect.

    Returns
    -------
    None
    """
    print("First 5 rows:")
    display(df.head())
    print("\nShape of the DataFrame:", df.shape)
    print("\nData info:")
    df.info()
    print("\nDescriptive statistics:")
    display(df.describe(include='all'))
    print("\nDuplicate rows:", df.duplicated().sum())
    print("\nDuplicate percentage:",
          (df.duplicated().sum()/len(df)*100).round(2), "%")
    print("\nMissing values:")
    display(pd.DataFrame({
        'values': df.isnull().sum(),
        'percentage': (df.isnull().sum()/len(df)*100).round(2)
    }))

### 2.1 Calls Dataset

In [4]:
# Display an overview of the calls DataFrame
data_overview(df_calls)

First 5 rows:


,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
0,166377,2019-08-04 00:00:00+03:00,in,False,NaN,True,2,0,4
1,166377,2019-08-05 00:00:00+03:00,out,True,880022.0,True,3,0,5
2,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,True,1,0,1
3,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,False,1,10,18
4,166377,2019-08-05 00:00:00+03:00,out,False,880022.0,True,3,0,25



Shape of the DataFrame: (53902, 9)

Data info:
<class 'pandas.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              53902 non-null  int64  
 1   date                 53902 non-null  str    
 2   direction            53902 non-null  str    
 3   internal             53785 non-null  object 
 4   operator_id          45730 non-null  float64
 5   is_missed_call       53902 non-null  bool   
 6   calls_count          53902 non-null  int64  
 7   call_duration        53902 non-null  int64  
 8   total_call_duration  53902 non-null  int64  
dtypes: bool(1), float64(1), int64(4), object(1), str(2)
memory usage: 3.3+ MB

Descriptive statistics:


,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
count,53902.000000,53902,53902,53785,45730.000000,53902,53902.000000,53902.000000,53902.000000
unique,NaN,119,2,2,NaN,2,NaN,NaN,NaN
top,NaN,2019-11-25 00:00:00+03:00,out,False,NaN,False,NaN,NaN,NaN
freq,NaN,1220,31917,47621,NaN,30334,NaN,NaN,NaN
mean,167295.344477,NaN,NaN,NaN,916535.993002,NaN,16.451245,866.684427,1157.133297
std,598.883775,NaN,NaN,NaN,21254.123136,NaN,62.917170,3731.791202,4403.468763
min,166377.000000,NaN,NaN,NaN,879896.000000,NaN,1.000000,0.000000,0.000000
25%,166782.000000,NaN,NaN,NaN,900788.000000,NaN,1.000000,0.000000,47.000000
50%,167162.000000,NaN,NaN,NaN,913938.000000,NaN,4.000000,38.000000,210.000000
75%,167819.000000,NaN,NaN,NaN,937708.000000,NaN,12.000000,572.000000,902.000000



Duplicate rows: 4900

Duplicate percentage: 9.09 %

Missing values:


,values,percentage
user_id,0,0.00
date,0,0.00
direction,0,0.00
internal,117,0.22
operator_id,8172,15.16
is_missed_call,0,0.00
calls_count,0,0.00
call_duration,0,0.00
total_call_duration,0,0.00


#### Calls dataset: first observations

- **Volume:** the calls dataset has 53,902 rows and 9 columns. Since each row aggregates several calls, the number of rows does not correspond to the number of calls.
- **Data types:** some columns need their type to be updated: `date` to datetime, `user_id` and `operator_id` to categorical, and `internal` to boolean after properly handling its missing values. `operator_id` is currently stored as float because of its missing values.
- **Identifiers:** the descriptive statistics of `user_id` and `operator_id` (mean, standard deviation, etc.) have no meaning, since these columns are identifiers.
- **Duplicates:** the dataset contains 4,900 fully duplicated rows (9.09%). Since each row already aggregates calls with the same characteristics, these duplicates are likely due to an error at data collection time and should be dropped.
- **Missing values in `operator_id`:** 15.16% of the rows have no operator. Since this is one of the key columns, these rows must be removed from the agent-level analysis.
- **Missing values in `internal`:** 0.22% of the rows have missing values. Given this low percentage, these rows can be removed without a meaningful impact on the analysis.
- **Overlap between issues:** some duplicated rows may also contain missing values, so missing values must be recounted after removing duplicates.
- **Dates:** `date` has 119 unique values and includes a time zone (+03:00). The period covered by the dataset must be checked.
- **Call direction:** `out` is the most frequent value (31,917 rows). This refers to rows, not calls, so the share of outbound calls must be calculated using `calls_count`.
- **Missed calls:** `False` is the most frequent value of `is_missed_call` (30,334 rows), so missed calls represent a large share of the rows. As with `direction`, the share of missed calls must be calculated using `calls_count`.
- **Distribution of `calls_count`:** the mean (16.45) is much higher than the median (4), and the maximum reaches 4,817, which indicates a right-skewed distribution.
- **Zero values in `call_duration`:** the 25th percentile is zero, which can be explained by the number of missed calls. This must be validated: if `is_missed_call` is True, `call_duration` should be zero.
- **Maximum durations:** the maximum values of `call_duration` (144,395) and `total_call_duration` (166,155) seem very high. The highest values must be checked against `calls_count`, using the duration per call, before deciding whether they are errors to be removed.
- **Consistency between durations:** since `total_call_duration` includes waiting time, it must always be greater than or equal to `call_duration`. This must be validated.